In [ ]:
from pathlib import Path
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
IMG_DIR = DATA_DIR / 'images'

In [ ]:
import pandas as pd

data = pd.read_csv(f'{DATA_DIR}/dataset_final.csv')
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
emotion_distribution = data['emotion'].value_counts()
plt.figure(figsize=(8, 5))
emotion_distribution.plot(kind='bar')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(emotion_distribution.describe())


utterance_lengths = data['utterance'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 5))
plt.hist(utterance_lengths, bins=20)
plt.title('Utterance Length Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
print(utterance_lengths.describe())

image_coverage = data['filename'].nunique()
print(f'Unique images: {image_coverage}')
actual_images_count = len(list(IMG_DIR.glob('*.jpg')))
print(f'Actual images in directory: {actual_images_count}')

invalid_images = set(data['filename']) - set(img.name for img in IMG_DIR.glob('*.jpg'))
print(f'Invalid image references: {len(invalid_images)}')

annotations_per_image = data.groupby('filename').size()
print(annotations_per_image.describe())

print(data.shape)

In [ ]:
from torchvision import transforms
from PIL import Image

def process_image(img_path):
    img = Image.open(img_path)
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
    ])
    img_tensor = preprocess(img)
    return img_tensor

print(process_image(f"{IMG_DIR}/{data['filename'].iloc[0]}").shape)

In [ ]:
from collections import Counter
MIN_FREQUENCY = 2

tokenized_utterances = data['utterance'].apply(lambda x: str(x).lower().split())
word_counts = Counter(word for utterance in tokenized_utterances for word in utterance)
vocab = {word for word in word_counts if word_counts[word] >= MIN_FREQUENCY}
vocab.update({'<PAD>', '<UNK>', '<SOS>', '<EOS>'})

word2idx = {word: idx for idx, word in enumerate(sorted(vocab))}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f'Vocabulary size: {len(vocab)}')
print(f'Index for special tokens: <PAD>={word2idx["<PAD>"]}, <UNK>={word2idx["<UNK>"]}, <SOS>={word2idx["<SOS>"]}, <EOS>={word2idx["<EOS>"]}')

sample_utterance = data['utterance'].iloc[0].lower()
tokenized = sample_utterance.split()
indices = [word2idx.get(word, word2idx['<UNK>']) for word in tokenized]
print(f'Original: {sample_utterance}')
print(f'Tokenized: {tokenized}')
print(f'Indices: {indices}')
reconstructed = ' '.join(idx2word[idx] for idx in indices)
print(f'Reconstructed: {reconstructed}')
